In [0]:
%run ../helpers/common_utilities

In [0]:
volume_location = f"/Volumes/{CATALOG_NAME}/{Default_schema}/{Volume_name}"
print(volume_location)

In [0]:
Silver_Table_dict = {
    'CNC':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.cnc_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/cnc_sensor_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_cnc_state_metrics",
        'query':"""
            
                    SELECT
                        sensor_event_id,
                        machine_id,
                        machine_type,
                        temperature,
                        vibration,
                        ae_rms,
                        rpm,
                        current,
                        runtime_hours,
                        power_kw,
                        alarm_count,
                        pressure,

                        CASE

                            -- FAULT
                            WHEN (
                                temperature > 90
                                OR vibration > 8
                                OR ae_rms > 10
                            )
                            THEN 'FAULT'

                            -- MAINTENANCE
                            WHEN (
                                MOD(runtime_hours, 500) < 5
                                AND rpm < 10
                                AND current < 1
                            )
                            THEN 'MAINTENANCE'

                            -- RUNNING
                            WHEN (
                                rpm >= 800
                                AND power_kw >= 5
                                AND current >= 5
                                AND temperature < 90
                            )
                            THEN 'RUNNING'

                            -- IDLE
                            WHEN (
                                rpm > 0
                                AND rpm < 800
                                AND power_kw > 0.5
                                AND power_kw < 5
                            )
                            THEN 'IDLE'

                            -- STOPPED
                            ELSE 'STOPPED'

                        END AS Machine_state,
                        publish_timestamp,
                        lead(publish_timestamp) over (order by publish_timestamp) as lead,
                        TIMESTAMPDIFF(
                            SECOND,
                            publish_timestamp,
                            lead
                        ) / 60.0 as timestamp_difference,
                        case 
                            when Machine_state = 'FAULT' then timestamp_difference else 0 
                        end 
                        as Unplanned_Downtime,
                    
                        case 
                            when Machine_state = 'MAINTENANCE' then timestamp_difference else 0 
                        end 
                        as planned_Downtime,

                        CASE
                                WHEN machine_state IN ('RUNNING', 'IDLE')
                                THEN timestamp_difference
                                ELSE 0
                        END AS healthy_time,


                        Unplanned_Downtime as error_time,

                        current_timestamp() as load_timestamp

                    FROM temp_view
                
                    """
    },

    'Motor':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.motor_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/motor_sensor_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_motor_state_metrics",
        'query':"""
            
                    SELECT
                        sensor_event_id,
                        machine_id,
                        machine_type,
                        temperature,
                        vibration,
                        ae_rms,
                        rpm,
                        current,
                        runtime_hours,
                        power_kw,
                        alarm_count,
                        pressure,

                         CASE

                            WHEN (

                                temperature >= 95
                                OR vibration >= 10
                                OR ae_rms >= 12

                            )

                            THEN 'FAULT'

                            WHEN (

                                MOD(runtime_hours, 1000) < 5
                                AND rpm < 50
                                AND current < 1

                            )

                            THEN 'MAINTENANCE'

                            WHEN (

                                rpm >= 1200
                                AND current >= 3
                                AND power_kw >= 3
                                AND temperature < 95

                            )

                            THEN 'RUNNING'

                            WHEN (

                                rpm > 0
                                AND rpm < 1200
                                AND power_kw > 0.2
                                AND power_kw < 3

                            )

                            THEN 'IDLE'

                            ELSE 'STOPPED'

                        END AS Machine_state,
                        publish_timestamp,
                        lead(publish_timestamp) over (order by publish_timestamp) as lead,
                        TIMESTAMPDIFF(
                            SECOND,
                            publish_timestamp,
                            lead
                        ) / 60.0 as timestamp_difference,
                        case 
                            when Machine_state = 'FAULT' then timestamp_difference else 0 
                        end 
                        as Unplanned_Downtime,
                    
                        case 
                            when Machine_state = 'MAINTENANCE' then timestamp_difference else 0 
                        end 
                        as planned_Downtime,

                        CASE
                                WHEN machine_state IN ('RUNNING', 'IDLE')
                                THEN timestamp_difference
                                ELSE 0
                        END AS healthy_time,


                        Unplanned_Downtime as error_time,

                        current_timestamp() as load_timestamp

                    FROM temp_view
                
                    """,
    },

    'Transformer':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.transformer_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/transformer_sensor_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_transformer_state_metrics",
        'query':"""
            
                    SELECT
                        sensor_event_id,
                        machine_id,
                        machine_type,
                        temperature,
                        vibration,
                        ae_rms,
                        rpm,
                        current,
                        runtime_hours,
                        power_kw,
                        alarm_count,
                        pressure,

                         CASE

                        -- =====================================================
                        -- FAULT
                        -- Abnormal overheating or internal issue
                        -- =====================================================
                        WHEN (
                            temperature >= 95
                            OR vibration >= 10
                            OR ae_rms >= 12
                        )
                        THEN 'FAULT'


                        -- =====================================================
                        -- MAINTENANCE
                        -- Scheduled maintenance / transformer isolated
                        -- =====================================================
                        WHEN (
                            MOD(runtime_hours, 1000) < 5
                            AND current < 0.5
                            AND power_kw < 0.2
                        )
                        THEN 'MAINTENANCE'


                        -- =====================================================
                        -- STOPPED
                        -- No load / de-energized
                        -- =====================================================
                        WHEN (
                            current < 0.5
                            AND power_kw < 0.2
                        )
                        THEN 'STOPPED'


                        -- =====================================================
                        -- RUNNING
                        -- Healthy loaded transformer
                        -- =====================================================
                        WHEN (
                            current >= 3
                            AND power_kw >= 3
                            AND temperature < 95
                        )
                        THEN 'RUNNING'

                        -- =====================================================
                        -- IDLE
                        -- Energized but lightly loaded
                        -- =====================================================
                        WHEN (
                            current > 0.5
                            AND current < 3
                            AND power_kw > 0.2
                            AND power_kw < 3
                        )
                        THEN 'IDLE'

                        ELSE 'STOPPED'

                    END AS Machine_state,
                        publish_timestamp,
                        lead(publish_timestamp) over (order by publish_timestamp) as lead,
                        TIMESTAMPDIFF(
                            SECOND,
                            publish_timestamp,
                            lead
                        ) / 60.0 as timestamp_difference,
                        case 
                            when Machine_state = 'FAULT' then timestamp_difference else 0 
                        end 
                        as Unplanned_Downtime,
                    
                        case 
                            when Machine_state = 'MAINTENANCE' then timestamp_difference else 0 
                        end 
                        as planned_Downtime,

                        CASE
                                WHEN machine_state IN ('RUNNING', 'IDLE')
                                THEN timestamp_difference
                                ELSE 0
                        END AS healthy_time,


                        Unplanned_Downtime as error_time,

                        current_timestamp() as load_timestamp

                    FROM temp_view
                
                    """
    }
}
from concurrent.futures import ThreadPoolExecutor

def run_stream_processor(table_key):
    print(f'started for {table_key}')
    config = Silver_Table_dict[table_key]
    print(f'''started for {table_key}
          source_table_name={config['source_table_name']},
        checkpoint_location={config['checkpoint_location']},
        target_table={config['target_table']},
        query={config['query']}
          ''')
    processor = StreamProcessor(
        source_table_name=config['source_table_name'],
        checkpoint_location=config['checkpoint_location'],
        target_table=config['target_table'],
        query=config['query']
    )
    df = processor.read_stream()
    query = processor.write_stream(
        df=df,
        output_mode="append",
        trigger_type='availableNow'
    )
    query.awaitTermination()

with ThreadPoolExecutor() as executor:
    executor.map(run_stream_processor, Silver_Table_dict.keys())
